In [1]:
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from glob import glob
import json
from tqdm import tqdm
from utils import compute_llm_judge_scores
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

os.makedirs("Outputs/LLMaaJ", exist_ok=True)

/home/sahajps/US_CNs/env_CNsUS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
files = glob("../Experiments/Postprocessed Outputs/*.json")
files.sort()

tweets = json.load(open("../Data/tweets.json", encoding="utf-8"))

human_notes = json.load(open("../Experiments/Postprocessed Outputs/human.json", encoding="utf-8"))

def process_file(f):
    if "human" in f:
        return
    
    scores = {}
    notes = json.load(open(f, encoding="utf-8"))
    notes = dict(list(notes.items()))

    for i in tqdm(notes, desc=f):
        if notes[i]=="NA":
            continue
        scores[i] = compute_llm_judge_scores(
            tweets[i]["text"],
            tweets[i]["createdAt"],
            human_notes[i],
            notes[i]
        )

    out_path = f.replace("../Experiments/Postprocessed Outputs/", "Outputs/LLMaaJ/")
    json.dump(scores, open(out_path, "w", encoding="utf-8"), indent=4)

with ProcessPoolExecutor(max_workers=7) as executor:
    futures = [executor.submit(process_file, f) for f in files]
    for _ in tqdm(as_completed(futures), total=len(futures)):
        pass

In [3]:
def return_mean_scores(scores):
    try:
        return scores.mean().round(3)
    except:
        return scores

scores = {}
for f in sorted(glob("Outputs/LLMaaJ/*.json")):
    scores_dic = json.load(open(f))
    scores_df = pd.DataFrame(scores_dic).T
    scores_dic = {i: return_mean_scores(scores_df[i]) for i in scores_df.columns}
    scores_dic["method"] = os.path.basename(f).replace(".json", "")
    scores[scores_dic["method"]] = scores_dic

sorted_scores = []
for m in ['supernote_lite_Llama-3.1-8B-Instruct', 'supernote_lite_Ministral-8B-Instruct-2410', 'supernote_lite_Qwen3-14B', 'supernote_lite_Apriel-Nemotron-15b-Thinker', 'supernote_lite_gpt-5-nano', 'supernote_lite_gemini-2.5-flash', 'web_search_gpt-5-nano_T1', 'web_search_gpt-5-nano_T2', 'web_search_gemini-2.5-flash_T1', 'web_search_gemini-2.5-flash_T2', 'web_search_grok-4_T1', 'web_search_sonar-deep-research_T1', 'our_gpt-5-nano_T2', 'our_gemini-2.5-flash_T2']:
    sorted_scores.append(scores[m])

df = pd.DataFrame(sorted_scores)
df = df[['method', 'functional_errors', 'claim_alignment', 'fact_alignment', 'completeness', 'helpfulness']]
df.to_csv("Outputs/LLMaaJ_scores.csv", index=False)

df

,method,functional_errors,claim_alignment,fact_alignment,completeness,helpfulness
0,supernote_lite_Llama-3.1-8B-Instruct,4.087,3.585,2.479,2.464,2.796
1,supernote_lite_Ministral-8B-Instruct-2410,4.083,3.589,2.487,2.355,2.672
2,supernote_lite_Qwen3-14B,3.974,3.728,2.615,2.487,2.838
3,supernote_lite_Apriel-Nemotron-15b-Thinker,3.879,3.709,2.540,2.528,2.811
4,supernote_lite_gpt-5-nano,4.053,3.668,2.634,2.506,2.860
5,supernote_lite_gemini-2.5-flash,4.358,3.898,2.906,2.687,3.128
6,web_search_gpt-5-nano_T1,4.516,3.383,2.488,2.459,2.961
7,web_search_gpt-5-nano_T2,4.496,3.340,2.465,2.426,2.984
8,web_search_gemini-2.5-flash_T1,4.242,3.334,2.465,2.418,2.809
9,web_search_gemini-2.5-flash_T2,4.127,3.315,2.465,2.388,2.772


In [4]:
def mean_difference_significance(list1, list2):
    x, y = np.array(list1), np.array(list2)

    _, p_value = scipy_stats.ttest_ind(x, y, equal_var=False)

    if p_value < 0.001:
        sig = "***"   # p < 0.001
    elif p_value < 0.01:
        sig = "**"    # p < 0.01
    elif p_value < 0.05:
        sig = "*"     # p < 0.05
    else:
        sig = "ns"    # not significant

    return p_value, sig

sig_test_pairs = [('web_search_gpt-5-nano_T1', 'web_search_gpt-5-nano_T2'),
                  ('supernote_lite_gpt-5-nano', 'web_search_gpt-5-nano_T2'),
                  ('web_search_gemini-2.5-flash_T1', 'web_search_gemini-2.5-flash_T2'),
                  ('supernote_lite_gemini-2.5-flash', 'web_search_gemini-2.5-flash_T2'),
                  ('web_search_gpt-5-nano_T2', 'our_gpt-5-nano_T2'),
                  ('supernote_lite_gpt-5-nano', 'our_gpt-5-nano_T2'),
                  ('web_search_gemini-2.5-flash_T2', 'our_gemini-2.5-flash_T2'),
                  ('supernote_lite_gemini-2.5-flash', 'our_gemini-2.5-flash_T2')]

for metric in ['functional_errors', 'claim_alignment', 'fact_alignment', 'completeness', 'helpfulness']:
    print(f"\nSignificance tests for {metric}:\n")
    for (m1, m2) in sig_test_pairs:
        list1 = json.load(open(f"Outputs/LLMaaJ/{m1}.json"))
        list1 = pd.DataFrame(list1).T[metric].to_list()
        list2 = json.load(open(f"Outputs/LLMaaJ/{m2}.json"))
        list2 = pd.DataFrame(list2).T[metric].to_list()
        p_value, sig = mean_difference_significance(list1, list2)
        print(f"{m1} vs {m2}: p-value = {p_value:.4f}, significance = {sig}")


Significance tests for functional_errors:

web_search_gpt-5-nano_T1 vs web_search_gpt-5-nano_T2: p-value = 0.5801, significance = ns
supernote_lite_gpt-5-nano vs web_search_gpt-5-nano_T2: p-value = 0.0000, significance = ***
web_search_gemini-2.5-flash_T1 vs web_search_gemini-2.5-flash_T2: p-value = 0.0541, significance = ns
supernote_lite_gemini-2.5-flash vs web_search_gemini-2.5-flash_T2: p-value = 0.0004, significance = ***
web_search_gpt-5-nano_T2 vs our_gpt-5-nano_T2: p-value = 0.0000, significance = ***
supernote_lite_gpt-5-nano vs our_gpt-5-nano_T2: p-value = 0.0005, significance = ***
web_search_gemini-2.5-flash_T2 vs our_gemini-2.5-flash_T2: p-value = 0.0000, significance = ***
supernote_lite_gemini-2.5-flash vs our_gemini-2.5-flash_T2: p-value = 0.0486, significance = *

Significance tests for claim_alignment:

web_search_gpt-5-nano_T1 vs web_search_gpt-5-nano_T2: p-value = 0.6333, significance = ns
supernote_lite_gpt-5-nano vs web_search_gpt-5-nano_T2: p-value = 0.0007, sig

In [5]:
dic_sn = json.load(open("../Experiments/Postprocessed Outputs/supernote_lite_gpt-5-nano.json", encoding="utf-8"))

scores = {}
for f in sorted(glob("Outputs/LLMaaJ/our*.json")):
    scores_dic = json.load(open(f))
    scores_w_c = {i: scores_dic[i] for i in scores_dic if dic_sn[i] != "NA"}
    scores_wo_c = {i: scores_dic[i] for i in scores_dic if dic_sn[i] == "NA"}

    scores_df_w_c = pd.DataFrame(scores_w_c).T
    scores_dic_w_c = {i: return_mean_scores(scores_df_w_c[i]) for i in scores_df_w_c.columns}
    scores_dic_w_c["method"] = os.path.basename(f).replace(".json", "") + "_with_context"
    scores[scores_dic_w_c["method"]] = scores_dic_w_c

    scores_df_wo_c = pd.DataFrame(scores_wo_c).T
    scores_dic_wo_c = {i: return_mean_scores(scores_df_wo_c[i]) for i in scores_df_wo_c.columns}
    scores_dic_wo_c["method"] = os.path.basename(f).replace(".json", "") + "_without_context"
    scores[scores_dic_wo_c["method"]] = scores_dic_wo_c

sorted_scores = []
for m in ['our_gpt-5-nano_T2_without_context', 'our_gpt-5-nano_T2_with_context', 'our_gemini-2.5-flash_T2_without_context', 'our_gemini-2.5-flash_T2_with_context']:
    sorted_scores.append(scores[m])

df = pd.DataFrame(sorted_scores)
df = df[['method', 'functional_errors', 'claim_alignment', 'fact_alignment', 'completeness', 'helpfulness']]
df.to_csv("Outputs/abl_scores_LLMaaJ.csv", index=False)

df

,method,functional_errors,claim_alignment,fact_alignment,completeness,helpfulness
0,our_gpt-5-nano_T2_without_context,4.238,3.578,2.565,2.623,3.049
1,our_gpt-5-nano_T2_with_context,4.291,3.743,2.732,2.826,3.219
2,our_gemini-2.5-flash_T2_without_context,4.486,3.568,2.532,2.605,2.991
3,our_gemini-2.5-flash_T2_with_context,4.470,3.890,2.807,2.860,3.216


In [6]:
sig_test = ['our_gpt-5-nano_T2', 'our_gemini-2.5-flash_T2']

for metric in ['functional_errors', 'claim_alignment', 'fact_alignment', 'completeness', 'helpfulness']:
    print(f"\nSignificance tests for {metric}:\n")
    for m in sig_test:
        full_list = json.load(open(f"Outputs/LLMaaJ/{m}.json"))
        list1 = {i: full_list[i] for i in full_list if dic_sn[i] != "NA"}
        list2 = {i: full_list[i] for i in full_list if dic_sn[i] == "NA"}

        list1 = pd.DataFrame(list1).T[metric].to_list()
        list2 = pd.DataFrame(list2).T[metric].to_list()
        p_value, sig = mean_difference_significance(    list1, list2)
        print(f"{m} with context vs without context: p-value = {p_value:.4f}, significance = {sig}")


Significance tests for functional_errors:

our_gpt-5-nano_T2 with context vs without context: p-value = 0.3490, significance = ns
our_gemini-2.5-flash_T2 with context vs without context: p-value = 0.7943, significance = ns

Significance tests for claim_alignment:

our_gpt-5-nano_T2 with context vs without context: p-value = 0.1572, significance = ns
our_gemini-2.5-flash_T2 with context vs without context: p-value = 0.0072, significance = **

Significance tests for fact_alignment:

our_gpt-5-nano_T2 with context vs without context: p-value = 0.1233, significance = ns
our_gemini-2.5-flash_T2 with context vs without context: p-value = 0.0161, significance = *

Significance tests for completeness:

our_gpt-5-nano_T2 with context vs without context: p-value = 0.0812, significance = ns
our_gemini-2.5-flash_T2 with context vs without context: p-value = 0.0275, significance = *

Significance tests for helpfulness:

our_gpt-5-nano_T2 with context vs without context: p-value = 0.1276, significa